In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np
import cv2
import time
import os
import sys
from pathlib import Path

# Add the challenge_solution to path
sys.path.insert(0, 'challenge_solution')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode
sys.path.insert(0, '/home/kevin.pasini/projet_explo/kevin/uqmodels/abench/')
import challenge_solution.df_utils as dm
from challenge_solution.torch_dataloader import ImageDataFrameDataset

# Exemple de transform basique Redimensionne et normalise
transform = transforms.Compose([transforms.Resize(size=(224, 224), interpolation=InterpolationMode.BILINEAR, max_size=None, antialias=True),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

#Genere un meta dataframe utilisé pour acceder au données.
dossier_des_images = "./datasets/welding-detection-challenge-dataset/"
df_data = dm.explore_csv_hierarchy(dossier_des_images,depth_name_list=['seam','decision','type_label'],allowed_ext='.jpeg')
df_data['path'] = df_data['path'].apply(lambda p: os.path.relpath(p, dossier_des_images))

# Mapping des labels
mapping = {'OK': 0, 'KO': 1}
df_data['label'] = df_data['decision'].map(mapping)

print("Types de décisions :", df_data['decision'].unique())
print("Valeurs manquantes :", df_data['label'].isna().sum())
print("Types de soudure :", df_data['type_label'].unique())

welding_types = ["c20", "c33", "c102"]

for wt in welding_types:
    print(f"\n=== Entraînement modèle pour {wt} ===")
    
    # Filtrer le dataset sur le type de soudure (seam)
    df_subset = df_data[df_data['seam'] == wt]
    
    if df_subset.empty:
        print(f"⚠️  Aucun échantillon trouvé pour {wt}, vérifie tes chemins et extensions.")
        continue

    # Stratified split
    df_train, df_val = dm.stratified_train_val_split(df_subset, ['seam','decision'], alpha=0.95, random_state=42)
    
    # Créer les datasets
    Train_Dataset = ImageDataFrameDataset(
        df=df_train,
        root_dir="./datasets/welding-detection-challenge-dataset/",  # chemin racine
        path_col="path",
        label_col="label",
        transform=transform,
        channels_first=True
    )
    
    Val_Dataset = ImageDataFrameDataset(
        df=df_val,
        root_dir="./datasets/welding-detection-challenge-dataset/",
        path_col="path",
        label_col="label",
        transform=transform,
        channels_first=True
    )

    # Initialiser et entraîner
    ai_component = MyAIComponent()
    ai_component.init_model()
    ai_component.train_model(
        Train_Dataset,
        Val_Dataset,
        device='cpu',  # ou 'cuda' si disponible
        save_path=f"best_model_{wt}.pth",
        augmentation_fn=None,
        preprocess_fn=None,
        epochs=1,
        batch_size=64,
        lr=3e-4
    )


print("\n🎉 Entraînement des trois modèles terminé !")


Types de décisions : ['KO' 'OK']
Valeurs manquantes : 0
Types de soudure : ['expert' 'operator']

=== Entraînement modèle pour c20 ===
🟦 Training started...


Training: 100%|██████████| 73/73 [09:11<00:00,  7.56s/it, loss=0.191] 


📘 Epoch 1/1 | Train: 0.2083 | Val: 0.1141
💾 Best model updated → best_model_c20.pth
🏁 Training completed in 569.8s

=== Entraînement modèle pour c33 ===
🟦 Training started...


Training: 100%|██████████| 133/133 [17:15<00:00,  7.79s/it, loss=0.201]  


📘 Epoch 1/1 | Train: 0.0972 | Val: 0.0214
💾 Best model updated → best_model_c33.pth
🏁 Training completed in 1084.1s

=== Entraînement modèle pour c102 ===
🟦 Training started...


Training: 100%|██████████| 133/133 [19:35<00:00,  8.84s/it, loss=0.00526]


📘 Epoch 1/1 | Train: 0.1026 | Val: 0.0304
💾 Best model updated → best_model_c102.pth
🏁 Training completed in 1196.1s

🎉 Entraînement des trois modèles terminé !


In [36]:
import os
import cv2
import numpy as np
from pathlib import Path
from challenge_solution.AIComponent import MyAIComponent

# --- Initialisation du composant AI ---
ai_component = MyAIComponent()

# --- Fonction utilitaire pour charger et convertir une image en RGB ---
def load_image(image_path: str) -> np.ndarray:
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Image introuvable : {image_path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# --- Fonction de prédiction pour une image et son type de soudure ---
def predict_welding_image(image_path: str, welding_type: str):
    # Charger l'image
    img = load_image(image_path)

    # Charger le modèle spécialisé correspondant
    ai_component.load_model(welding_type)  # <-- IMPORTANT !

    # Préparer les métadonnées
    metadata = [{"type_label": welding_type}]

    # Faire la prédiction via le modèle spécialisé
    results = ai_component.predict([img], metadata)

    # Extraire résultats
    pred = results['predictions'][0]
    probs = results['probabilities'][0]
    ood = results['OOD_scores'][0]

    # Affichage clair
    print(f"\n🔹 Image : {image_path}")
    print(f"  Modèle utilisé : {welding_type}")
    print(f"  Prediction : {pred}")
    print(f"  Probabilités : OK={probs[0]:.3f}, KO={probs[1]:.3f}, UNKNOWN={probs[2]:.3f}")
    print(f"  OOD score : {ood:.3f}")

# --- Exemple d'utilisation ---
#image_path = "./datasets/welding-detection-challenge-dataset/c20/KO/expert/sample_357.jpeg"
image_path = "./datasets/welding-detection-challenge-dataset/c33/OK/expert/sample_10.jpeg"


predict_welding_image(image_path, welding_type="c33")  # c20 / c33 / c102 selon le type de soudure


🔧 Loading Welding Quality AI Component...
Missing keys (dans le modèle mais pas dans le checkpoint) : []
Unexpected keys (dans le checkpoint mais pas dans le modèle) : []
✅ Model weights loaded from C:\Users\phile\Documents\Cours\ING5\IA de confiance\Challenge_welding\challenge_solution\best_model.pth
🔧 Using CPU
🔥 Model warmup completed
✅ AI Component loaded on cpu

🔹 Image : ./datasets/welding-detection-challenge-dataset/c33/OK/expert/sample_10.jpeg
  Modèle utilisé : c33
  Prediction : OK
  Probabilités : OK=0.990, KO=0.010, UNKNOWN=0.000
  OOD score : 0.000
